# Orchestrator: Dynamic Payload Estimation using PINNs

This notebook serves as the main entry point to execute and evaluate Physics-Informed Neural Networks (PINNs) developed for real-time payload (mass) estimation.

## Evaluated Methodologies (Peugeot Partner):
1. **Direct Mechanical Model (Suspension Sensor):** Estimates mass by analyzing vertical suspension deflection under quadratic stiffness constraints. (Successful empirical model, $R^2 \approx 0.94$).
2. **Indirect Thermodynamic Model (OBD Engine Data):** Resolves longitudinal power balance equations based on fuel consumption telemetry.

All physical models, neural network structures, and training engines are modularized in local Python scripts.

In [ ]:
import sys
import os
# Add src directory to system path to load modular files
sys.path.append(os.path.join(os.getcwd(), 'src'))

import pandas as pd
import numpy as np

# Import modular utilities from our repository
from data_utils import cargar_y_limpiar_sensor, cargar_y_limpiar_obd_peugeot
from train import entrenar_suspension_pinn, entrenar_termo_pinn
from visualizations import imprimir_reporte_metricas, graficar_estimacion_vs_real, graficar_distribucion_errores

## 1. Direct Mechanical Model (Suspension Sensor) - Peugeot Partner

This model uses vertical suspension deflection ($x$) to optimize mass based on a quadratic stiffness spring equation:

$$M \frac{d^2x}{dt^2} + C \frac{dx}{dt} + (K_1 \cdot x + K_2 \cdot x^2) - (M - M_{base}) g = 0$$

The neural network retains weights and optimized mass parameters incrementally across windows, providing high stability and accuracy.

In [ ]:
# Calibrated Suspension and Physical Constants
M_BASE = 1301.0       # Vehicle base tare + driver (kg)
SENSOR_VACIO = 162.42 # Zero-payload sensor calibration offset (mm)
C_AMORT = 6537.0      # Suspension damping coefficient (Ns/m)
K1 = 84705.20         # Suspension linear stiffness coefficient (N/m)
K2 = 2903352.28       # Suspension quadratic stiffness coefficient (N/m²)

# Load and clean suspension telemetry data
df_sensor = cargar_y_limpiar_sensor(
    file_path='data/09 OBD Peugeot Datos Limpios(Sheet1).csv',
    sensor_vacio=SENSOR_VACIO,
    smoothing_window=2,
    speed_threshold=1.0
)

# Train the Suspension PINN (Incremental Weights / Mass)
tiempos_sens, y_real_sens, y_pred_sens = entrenar_suspension_pinn(
    df=df_sensor,
    m_base=M_BASE,
    c_amort=C_AMORT,
    k1=K1,
    k2=K2,
    tam_ventana=100,
    epocas_init=1500,
    epocas_normal=150,
    epocas_jump=1000,
    lr_net=1e-3,
    lr_masa=10.0,
    optimize_k=False
)

# Compute metrics
imprimir_reporte_metricas(y_real_sens, y_pred_sens, "PINN Suspension (Peugeot Partner)")

In [ ]:
# Plot payload estimation vs ground truth over time
graficar_estimacion_vs_real(
    tiempos=tiempos_sens,
    y_real=y_real_sens,
    y_pred=y_pred_sens,
    titulo="Dynamic Payload Estimation (Suspension Sensor) - Peugeot Partner",
    color_pred="orange"
)

# Plot error density distribution
graficar_distribucion_errores(
    y_real=y_real_sens,
    y_pred=y_pred_sens,
    titulo="Error Distribution: PINN with Suspension Sensor"
)

## 2. Indirect Thermodynamic Model (OBD Engine Data) - Peugeot Partner

This model balances vehicle required traction power ($P_{req}$) with available chemical combustion power ($P_{disp}$):

$$P_{req} = \max(0, F_{req} \cdot V)$$
$$P_{disp} = \rho_{fuel} \cdot (\text{Fuel Flow} - \text{Idle Burn}) \cdot LHV_f \cdot \eta$$

It utilizes a dynamic acceleration mask and segment memory inheritance to optimize mass.

In [ ]:
# Vehicle and Environment Constants
M_BASE = 1301.0
ETA_ESTATICO = 0.12    # Thermal engine efficiency
LHV_F = 43000000.0     # Lower heating value of gasoline (J/kg)
RHO_F = 0.750          # Density of gasoline (kg/L)
A_F = 2.5              # Frontal aerodynamic area (m²)
C_D = 0.35             # Aerodynamic drag coefficient
F_R = 0.01             # Rolling resistance coefficient
RHO_AIR = 1.17         # Air density in Monterrey (kg/m³)
IDLE_BURN = 0.0018     # Engine idle fuel rate (L/s)

# Load and clean OBD data
df_obd_peugeot = cargar_y_limpiar_obd_peugeot('data/09 OBD Peugeot Datos Limpios(Sheet1) - V2.csv')

# Train the Thermodynamic PINN
tiempos_obd_p, y_real_obd_p, y_pred_obd_p = entrenar_termo_pinn(
    df=df_obd_peugeot,
    m_base=M_BASE,
    m_max=2500.0,
    eta=ETA_ESTATICO,
    lhv_f=LHV_F,
    rho_f=RHO_F,
    c_d=C_D,
    a_f=A_F,
    f_r=F_R,
    rho_air=RHO_AIR,
    idle_burn=IDLE_BURN,
    vehiculo="peugeot",
    tam_ventana=60,
    epocas_init=500,
    epocas_normal=500,
    lr_net=1e-3,
    lr_masa=1.0,
    use_mask=True,
    use_memory=True,
    warmup_epochs=300,
    peso_fisica=10.0,
    usar_huber=True
)

imprimir_reporte_metricas(y_real_obd_p, y_pred_obd_p, "PINN Thermodynamic OBD (Peugeot Partner)")

In [ ]:
# Plot OBD payload estimation vs ground truth over time
graficar_estimacion_vs_real(
    tiempos=tiempos_obd_p,
    y_real=y_real_obd_p,
    y_pred=y_pred_obd_p,
    titulo="Dynamic Payload Estimation (OBD Fuel Consumption) - Peugeot Partner",
    color_pred="green"
)